In [36]:
import pandas as pd
import numpy as np
from scipy import stats

In [37]:
df = pd.read_csv("kuesioner.csv", sep=";")

df.head()

,x1,x2,x3,x4,x5,y1,y2,y3,y4,y5
0,5,4,3,5,4,5,5,2,5,3
1,4,4,4,3,3,2,4,5,3,3
2,4,4,3,4,4,4,4,4,4,4
3,4,1,1,2,1,4,4,2,2,2
4,2,5,2,5,5,4,2,5,5,3


In [38]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   x1      30 non-null     int64
 1   x2      30 non-null     int64
 2   x3      30 non-null     int64
 3   x4      30 non-null     int64
 4   x5      30 non-null     int64
 5   y1      30 non-null     int64
 6   y2      30 non-null     int64
 7   y3      30 non-null     int64
 8   y4      30 non-null     int64
 9   y5      30 non-null     int64
dtypes: int64(10)
memory usage: 2.5 KB


In [39]:
x_items = ["x1", "x2", "x3", "x4", "x5"]
y_items = ["y1", "y2", "y3", "y4", "y5"]

In [40]:
n = len(df)

print("Jumlah responden:", n)

Jumlah responden: 30


# Uji validitas

## Hipotesis

H0: $\rho$ = 0 (Tidak ada korelasi yang signifikan atau item pertanyaan tidak valid.) <br>
H1: $\rho$ $\neq$ 0 (Ada korelasi yang signifikan atau item pertanyaan valid.)

## Nilai kritis

In [41]:
alpha = 0.05
dfree = n - 2

t_tabel = stats.t.ppf(1 - alpha / 2, dfree)

r_tabel = t_tabel / np.sqrt(t_tabel**2 + dfree)

print("n =", n)
print("df =", dfree)
print("r_tabel =", round(r_tabel, 4))

n = 30
df = 28
r_tabel = 0.361


## Statistik uji

Menggunakan korelasi Pearson Product Moment dan tingkat signifikansi.

### Uji validitas X

In [42]:
hasil_x = []

for item in x_items:
    item_score = df[item]
    total_score = df[x_items].drop(columns=item).sum(axis=1)

    r_hitung, p_value = stats.pearsonr(item_score, total_score)

    hasil_x.append({
        "Item": item,
        "r_hitung": r_hitung,
        "p_value": p_value
    })

hasil_x = pd.DataFrame(hasil_x)

hasil_x

,Item,r_hitung,p_value
0,x1,0.350397,0.057652
1,x2,0.744765,0.000002
2,x3,0.350054,0.057913
3,x4,0.417135,0.021829
4,x5,0.588477,0.000625


### Uji validitas Y

In [43]:
hasil_y = []

for item in y_items:
    item_score = df[item]
    total_score = df[y_items].drop(columns=item).sum(axis=1)

    r_hitung, p_value = stats.pearsonr(item_score, total_score)

    hasil_y.append({
        "Item": item,
        "r_hitung": r_hitung,
        "p_value": p_value
    })

hasil_y = pd.DataFrame(hasil_y)

hasil_y

,Item,r_hitung,p_value
0,y1,0.509693,0.004014
1,y2,0.476316,0.007794
2,y3,0.302242,0.104519
3,y4,0.611287,0.000332
4,y5,0.524435,0.002930


## Keputusan

In [44]:
hasil_x["Keputusan"] = np.where(
    hasil_x["r_hitung"] > r_tabel,
    "Valid",
    "Tidak Valid"
)

hasil_y["Keputusan"] = np.where(
    hasil_y["r_hitung"] > r_tabel,
    "Valid",
    "Tidak Valid"
)

## Kesimpulan

In [45]:
print("Uji validitas X: Kepadatan kampus")
display(hasil_x)

print("\nUji Y: Pengalaman akademik")
display(hasil_y)

Uji validitas X: Kepadatan kampus


,Item,r_hitung,p_value,Keputusan
0,x1,0.350397,0.057652,Tidak Valid
1,x2,0.744765,0.000002,Valid
2,x3,0.350054,0.057913,Tidak Valid
3,x4,0.417135,0.021829,Valid
4,x5,0.588477,0.000625,Valid



Uji Y: Pengalaman akademik


,Item,r_hitung,p_value,Keputusan
0,y1,0.509693,0.004014,Valid
1,y2,0.476316,0.007794,Valid
2,y3,0.302242,0.104519,Tidak Valid
3,y4,0.611287,0.000332,Valid
4,y5,0.524435,0.002930,Valid


# Uji reliabilitas

In [46]:
# Cronbach's Alpha
def cronbach_alpha(df_items):
    k = df_items.shape[1]
    if k < 2:
        return 0
    
    varians_item = df_items.var(ddof=1).sum()
    varians_total = df_items.sum(axis=1).var(ddof=1)
    
    return (k / (k - 1)) * (1 - (varians_item / varians_total))

# cari item valid, p-value < 0.05
def get_item_valid(df_lengkap, prefix_kolom):
    semua_item = [col for col in df_lengkap.columns if col.startswith(prefix_kolom)]
    skor_total = df_lengkap[semua_item].sum(axis=1)
    
    item_valid = []
    for col in semua_item:
        r_hitung, p_value = stats.pearsonr(df_lengkap[col], skor_total)
        if p_value < 0.05:
            item_valid.append(col)
            
    return item_valid

## Uji reliabilitas variabel X

In [47]:
item_valid_x = get_item_valid(df, 'x')
alpha_x = cronbach_alpha(df[item_valid_x])

print("Uji reliabilitas variabel X")
print(f"Item yang Valid : {item_valid_x}")
print(f"Cronbach's Alpha: {alpha_x:.4f}")
print(f"Keputusan       : {'Reliabel' if alpha_x > 0.60 else 'Tidak Reliabel'}")

print("\n")

Uji reliabilitas variabel X
Item yang Valid : ['x1', 'x2', 'x3', 'x4', 'x5']
Cronbach's Alpha: 0.7214
Keputusan       : Reliabel




## Uji reliabilitas variabel Y

In [48]:
item_valid_y = get_item_valid(df, 'y')
alpha_y = cronbach_alpha(df[item_valid_y])

print("Uji reliabilitas variabel Y")
print(f"Item yang valid : {item_valid_y}")
print(f"Cronbach's Alpha: {alpha_y:.4f}")
print(f"Keputusan       : {'Reliabel' if alpha_y > 0.60 else 'Tidak Reliabel'}")

Uji reliabilitas variabel Y
Item yang valid : ['y1', 'y2', 'y3', 'y4', 'y5']
Cronbach's Alpha: 0.7196
Keputusan       : Reliabel
